# Module 33 — The client, the handshake, and runtime discovery

**THE ONE IDEA:** an MCP client does not know what a server can do until it **asks**. That
is the whole point — tools are discovered **at runtime**, not compiled into your agent.

```
initialize            client and server exchange protocol version + capabilities
  -> initialized      notification; the session is now open
tools/list            DISCOVERY — the agent learns the tool surface here
tools/call            operation
shutdown              teardown
```

Contrast with every module before this one: `_tools.py` schemas were **imported**. Your
agent had to be redeployed to gain a tool. Here a server can be swapped and the agent
picks up its new tools on the next handshake.

No `mcp` package, no API key.


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import json
PROTOCOL = "2025-06-18"          # the version string is NEGOTIATED, not assumed

class Server:
    NAME, VERSION = "bank-policy-server", "0.1.0"
    CAPS = {"tools": {"listChanged": True}, "resources": {}, "prompts": {}}
    TOOLS = {"search_policy": {"description": "Search bank policy.",
                               "inputSchema": {"type": "object",
                                               "properties": {"query": {"type": "string"}},
                                               "required": ["query"]}}}
    def __init__(self): self.initialized = False

    def handle(self, req):
        m, p, i = req.get("method"), req.get("params", {}), req.get("id")
        if m == "initialize":
            if p.get("protocolVersion") != PROTOCOL:
                return {"jsonrpc": "2.0", "id": i, "error": {"code": -32602,
                        "message": f"unsupported protocol {p.get('protocolVersion')}"}}
            return {"jsonrpc": "2.0", "id": i, "result": {
                "protocolVersion": PROTOCOL, "capabilities": self.CAPS,
                "serverInfo": {"name": self.NAME, "version": self.VERSION}}}
        if m == "notifications/initialized":
            self.initialized = True; return None            # notifications have NO id
        if not self.initialized:                            # ORDER IS ENFORCED
            return {"jsonrpc": "2.0", "id": i, "error": {"code": -32002,
                    "message": "server not initialized"}}
        if m == "tools/list":
            return {"jsonrpc": "2.0", "id": i, "result": {"tools": [
                {"name": n, **t} for n, t in self.TOOLS.items()]}}
        return {"jsonrpc": "2.0", "id": i,
                "error": {"code": -32601, "message": f"method not found: {m}"}}

## The handshake, step by step

In [ ]:
srv, n = Server(), [0]
def send(method, params=None, notify=False):
    n[0] += 1
    req = {"jsonrpc": "2.0", "method": method, "params": params or {}}
    if not notify: req["id"] = n[0]
    res = srv.handle(req)
    tag = "notify" if notify else f"id={req.get('id')}"
    print(f"  -> {method:28} ({tag})")
    if res: print(f"  <- {json.dumps(res.get('result', res.get('error')))[:96]}")
    return res

print("1. OPERATION BEFORE HANDSHAKE — should be refused:")
send("tools/list")

print("\n2. initialize:")
send("initialize", {"protocolVersion": PROTOCOL,
                    "capabilities": {"roots": {}},
                    "clientInfo": {"name": "ladder-client", "version": "0.1"}})
print("\n3. initialized notification:")
send("notifications/initialized", notify=True)

## Runtime discovery — the agent learns its own tool surface

In [ ]:
res = send("tools/list")
discovered = res["result"]["tools"]

# Convert MCP tool defs into the OpenAI envelope module 08 expects.
# This adapter is ALL that langchain-mcp-adapters does.
def to_openai(tools):
    return [{"type": "function", "function": {"name": t["name"],
             "description": t["description"], "parameters": t["inputSchema"]}}
            for t in tools]

print("\ndiscovered at runtime, not imported:")
print(json.dumps(to_openai(discovered), indent=1)[:340])

## Version negotiation fails loudly

In [ ]:
bad = Server()
r = bad.handle({"jsonrpc": "2.0", "id": 1, "method": "initialize",
                "params": {"protocolVersion": "1999-01-01"}})
print("mismatched version ->", r["error"]["message"])

print("""
LESSON - four things the lifecycle buys you, in order of how much they matter:

  RUNTIME DISCOVERY   tools/list is why MCP exists. Every earlier module IMPORTED
                      its schemas from _tools.py, so gaining a tool meant
                      redeploying the agent. Here the server is swapped and the
                      agent learns the new surface on the next handshake. That is
                      what turns M hosts x N tools into M + N.

  NEGOTIATION         both sides state a protocol version and their capabilities.
                      A mismatch fails at the handshake with a clear error rather
                      than as a confusing failure three calls later.

  ENFORCED ORDER      operations before `initialized` are refused (-32002). The
                      session is a state machine, not a bag of endpoints.

  NOTIFICATIONS       messages with NO id expect no reply. `initialized` is one;
                      so is `listChanged`, which lets a server tell the client its
                      tool list has changed mid-session - dynamic tool surfaces.

The adapter above is worth noticing: eight lines convert an MCP tool definition
into module 08's OpenAI envelope. langchain-mcp-adapters is not doing anything
you could not write. The VALUE IS THE AGREEMENT, not the code.""")

---

**Next:** `34_mcp_transports.ipynb`